In [5]:
from PIL import Image
import io
import pymupdf
import os
import json
import pytesseract
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"


# doc = pymupdf.open("../examples/603972-june-2023-question-paper-11.pdf")
doc = pymupdf.open("../examples/Test.pdf")

layout_data = []

In [6]:
# visualize image
def visualize_image(image_bytes):
    # convert binary image data into a file-like object and load it 
    image = Image.open(io.BytesIO(image_bytes))
    image.show()

# check if image contains text, distnguish between decorative/text-based image
def is_text_image(image_bytes):
    image = Image.open(io.BytesIO(image_bytes))
    # perform OCR on image to extract data
    data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)
    
    # check if exist text by confidence score
    confidence_scores = []
    for i, word in enumerate(data["text"]):
        if word.strip():    # only count actual words
            confidence_raw = data["conf"][i]
            try:
                confidence_score = int(confidence_raw)
            except:
                confidence_score = -1
            if confidence_score > 0:
                confidence_scores.append(confidence_score)
                
    if len(confidence_scores) == 0:
        return False
    
    average_confidence = sum(confidence_scores) / len(confidence_scores)
    return average_confidence > 50 

# get text details in span-level
def get_spans(page):
    layout_data = []
    # get metadata of text blocks
    page_dict = page.get_text("dict")
    
    for block in page_dict["blocks"]:
        if block["type"] == 0:      # 0: text, 1: image (embedded and inline)
            block_text = ""
            spans_data = []         # a piece of text with the same style      
            for line in block["lines"]:
                for span in line["spans"]:
                    block_text += span["text"]
                    # tokens = get_tokens(page, span)
                    spans_data.append({
                        "text": span["text"],
                        "font_name": span["font"],  # 1 = italic, 2 = bold, 4 = monospace, 8 = serif, 16 = symbolic
                        "font_size": span["size"],
                        "font_flags": span["flags"],
                        "color": span["color"],
                        "bbox": span["bbox"],
                        # "tokens": tokens
                    })
            layout_data.append({
                "type": "text",
                "bbox": block["bbox"],
                "content": block_text.strip(),
                "spans": spans_data
            })
    return layout_data
            
# get tokens from spans
def get_tokens(page, span):
    # page width and height for padding
    page_width = page.rect.width
    page_height = page.rect.height
    
    # padding avoids cutting off characters, improves OCR recognition, avoids rounding issues
    padding = max(2, span["size"]*0.3)
    span_bbox = span["bbox"]
    # bounding box of span with padding
    x0 = max(0, span_bbox[0] - padding)
    y0 = max(0, span_bbox[1] - padding)
    x1 = min(page_width, span_bbox[2] + padding)
    y1 = min(page_height, span_bbox[3] + padding)
    bbox = (x0, y0, x1, y1)
    
    # get tokens inside image
    pix = page.get_pixmap(clip=pymupdf.Rect(bbox), dpi=300)
    image_bytes = pix.tobytes("png")
    ocr_tokens = get_ocr_image(image_bytes)
    
    # convert back to pdf coordinates
    tokens_pdf = []
    span_width = x1 - x0
    span_height = y1 - y0
    image_width = pix.width
    image_height = pix.height
    
    for ocr_token in ocr_tokens:
        token_x0_pdf = x0 + (ocr_token["x0"] / image_width) * span_width
        token_y0_pdf = y0 + (ocr_token["y0"] / image_height) * span_height
        token_x1_pdf = x0 + ((ocr_token["x0"] + ocr_token["width"]) / image_width) * span_width
        token_y1_pdf = y0 + ((ocr_token["y0"] + ocr_token["height"]) / image_height) * span_height
        
        tokens_pdf.append({
            "bbox": [token_x0_pdf, token_y0_pdf, token_x1_pdf, token_y1_pdf],
            "content": ocr_token["content"],
            "conf": ocr_token["conf"]
        })
        
    return tokens_pdf

# get metadata of texts inside images
def get_ocr_image(image_bytes):
    image = Image.open(io.BytesIO(image_bytes))
    # perform OCR on image to extract metadata of text
    data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)
     
    # iterate through visible text (some can be empty string but still have bouding box data and confidence score)
    # get left, top, width, height, conf, text of word blocks inside image
    word_list = []
    for i, word in enumerate(data["text"]):
        if word.strip():
            x0 = data["left"][i]
            y0 = data["top"][i]
            width = data["width"][i]
            height = data["height"][i]
            conf_raw = data["conf"][i]      # confidence score
            try:
                conf = int(conf_raw)
            except:
                conf = -1
            
            word_dict = {
                "x0": x0,
                "y0": y0,
                "width": width,
                "height": height,
                "conf": conf,
                "content": word.strip() 
            }
            word_list.append(word_dict)
    return word_list


output_dir = "../data_layout"

# save metadata to json
def save_metadata(page_index, layout_data):
    # create "layouts" folder inside output_dir
    page_folder = os.path.join(output_dir, f"page_{page_index}")
    os.makedirs(page_folder, exist_ok=True)
    # define path for JSON files
    json_path = os.path.join(page_folder, f"page_{page_index}_layout.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(layout_data, f, ensure_ascii=False, indent=2)

# create and return image file path
os.makedirs(output_dir, exist_ok=True)
def save_image_file(image_bytes, ext, page_index, image_index):
    page_folder = os.path.join(output_dir, f"page_{page_index}", "images")
    os.makedirs(page_folder, exist_ok=True)
    file_path = os.path.join(page_folder, f"img_{image_index}.{ext}")
    with open(file_path, "wb") as f:
        f.write(image_bytes)
    return file_path
    
# save page as pixmap, used as input for LayoutMv3
def save_page_as_pixmap(page, page_index):
    page_pixmap = page.get_pixmap(dpi=300)
    page_folder = os.path.join(output_dir, f"page_{page_index}")
    os.makedirs(page_folder, exist_ok=True)
    file_path = os.path.join(page_folder, f"page_{page_index}_image.png")
    page_pixmap.save(file_path)
    return file_path

In [7]:
import shutil

# Iterate over all files and subdirectories in the folder
folder_path = "../data_layout"
for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)  # remove file or link
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)  # remove directory and its contents
    except Exception as e:
        print(f"Failed to delete {file_path}. Reason: {e}")

# process each page
for page_index, page in enumerate(doc):
    layout_data = []
    
    # get pixmap object (not PNG data)
    save_page_as_pixmap(page, page_index)
    
    # get metadata of text blocks
    spans = get_spans(page)
    layout_data.extend(spans) 
    
    # gett metadata of image blocks
    image_blocks = page.get_image_info()
    for image_index, image_block in enumerate(image_blocks):
        bbox = image_block["bbox"]
        xref = image_block.get("xref", image_block.get("image"))    # if old version: xref, if new version: image_block.get("image")
        
        # embedded contains 'xref', inline does not
        if xref:
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]   # binary data
            ext = base_image["ext"]
        else:
            # Extract inline image with high DPI for better quality
            # Higher DPI (600) produces sharper images in DOCX output
            # Note: This only affects inline images; embedded images (with xref) 
            # are extracted at their original resolution from the PDF
            pix = page.get_pixmap(clip=pymupdf.Rect(bbox), dpi=600)  # pixel image
            image_bytes = pix.tobytes("png")
            ext = "png"
            
        image_path = save_image_file(image_bytes, ext, page_index, image_index)
        
        # All images are treated as type "image"
        layout_data.append({
            "type": "image",
            "bbox": bbox,
            "xref": xref,
            "ext": ext
        })
    
    save_metadata(page_index, layout_data)